# Advanced Analysis: IV Gaussian Filter Data (b_scans)

This notebook performs comprehensive analysis on the Gaussian-filtered IV H-scan data.

**Note:** Temperature is extracted from the DataFrame as the single source of truth.

## Analysis Components:
1. I(H) curves at fixed voltages
2. IV characteristics at different magnetic fields
3. TMR ratio vs voltage
4. Quality metrics

## 1. Setup & Data Loading

In [ ]:
# Notebook setup
from scripts.utils import setup_notebook
PROJECT_ROOT, np, pd, plt, Path = setup_notebook()

# Additional imports
from scipy.interpolate import griddata
from scripts.IV_Hscan_gaussian import load_dataframe, get_current_at_voltage, get_asymmetric_current_at_voltage

In [ ]:
# Load DataFrame
df_path = PROJECT_ROOT / r"output/IV_H_scans/dataframes/b_scans/IV_gaussian_7K.pkl"
# Find the file matching the pattern
import glob
matching_files = glob.glob(str(df_path))
if not matching_files:
    raise FileNotFoundError(f"No dataframe file found matching pattern: {df_path}")
df_path = Path(matching_files[0])
df = load_dataframe(df_path)

# Extract temperature from DataFrame as SINGLE SOURCE OF TRUTH
if 'temperature' in df.columns:
    temperature = int(df['temperature'].iloc[0])
else:
    raise ValueError("Temperature column not found in DataFrame!")

print(f"\n{'='*60}")
print(f"DataFrame loaded successfully!")
print(f"Temperature (from DataFrame): {temperature} K")
print(f"{'='*60}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"H field range: {df['H'].min():.4f} to {df['H'].max():.4f} T")
print(f"Number of measurements: {len(df)}")

## 2. I(H) Curves at Fixed Voltages

Plot current vs magnetic field at specific bias voltages.

In [ ]:
import matplotlib.cm as cm

voltages_to_plot = np.linspace(-1, 1, 51)
print(f"Will plot I(H) curves at {len(voltages_to_plot)} voltages.")

cmap = cm.get_cmap('RdBu_r')
norm = plt.Normalize(vmin=voltages_to_plot.min(), vmax=voltages_to_plot.max())

fig, ax = plt.subplots(figsize=(12, 7))

for V_val in voltages_to_plot:
    I_vs_H = get_current_at_voltage(df, V_val, use_filtered=True)
    color = cmap(norm(V_val))
    ax.plot(I_vs_H['H'], I_vs_H['I_at_V'] * 1e6, 'o-', color=color, markersize=4, linewidth=2, alpha=0.8)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('Voltage (V)', fontsize=14)

ax.set_xlabel('Magnetic Field H (T)', fontsize=14)
ax.set_ylabel('Current (µA)', fontsize=14)
ax.set_title(f'I(H) Curves at Fixed Voltages - {temperature}K', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. TMR Ratio Calculation

In [ ]:
voltage_points = np.linspace(-1.0, 1.0, 200)

H_low_threshold = 0.1   # T - low field region
H_high_threshold = 0.5  # T - high field region

tmr_values = []
tmr_errors = []

for V_target in voltage_points:
    data = get_current_at_voltage(df, V_target, use_filtered=True)
    H_vals = data['H'].values
    I_vals = np.abs(data['I_at_V'].values)
    
    low_field_mask = np.abs(H_vals) < H_low_threshold
    if low_field_mask.sum() > 0:
        currents_at_low_field = I_vals[low_field_mask]
        I_min = np.mean(currents_at_low_field)
        I_min_err = np.std(currents_at_low_field)/np.sqrt(len(currents_at_low_field)-1)
    else:
        I_min, I_min_err = np.nan, np.nan
    
    high_field_mask = np.abs(H_vals) > H_high_threshold
    if high_field_mask.sum() > 0:
        currents_at_high_field = I_vals[high_field_mask]
        I_max = np.mean(currents_at_high_field)
        I_max_err = np.std(currents_at_high_field)/np.sqrt(len(currents_at_high_field)-1)
    else:
        I_max, I_max_err = np.nan, np.nan
    
    if I_min > 0 and I_max > 0 and not np.isnan(I_min) and not np.isnan(I_max):
        tmr = I_max / I_min
        tmr_error = tmr * np.sqrt((I_max_err / I_max)**2 + (I_min_err / I_min)**2)
        tmr_values.append(tmr)
        tmr_errors.append(tmr_error)
    else:
        tmr_values.append(np.nan)
        tmr_errors.append(np.nan)

tmr_values = np.array(tmr_values)
tmr_errors = np.array(tmr_errors)

print(f"TMR calculation complete!")
print(f"Voltage range: {voltage_points.min():.2f} to {voltage_points.max():.2f} V")
print(f"TMR ratio range: {np.nanmin(tmr_values):.3f} to {np.nanmax(tmr_values):.3f}")

## 4. Plot TMR Ratio vs Voltage

In [ ]:
plt.figure(figsize=(12, 7))

plt.plot(voltage_points, tmr_values, '-', color='dodgerblue', linewidth=2, label='TMR Ratio')
plt.fill_between(voltage_points, tmr_values - tmr_errors, tmr_values + tmr_errors, color='dodgerblue', alpha=0.2, label='Propagated Error')

plt.xlabel("Voltage (V)", fontsize=14)
plt.ylabel("TMR Ratio (I_max / I_min)", fontsize=14)
plt.title(f"TMR Ratio vs. Applied Voltage - {temperature}K", fontsize=16)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)
plt.ylim(bottom=0)
plt.tight_layout()
plt.show()

## 5. Save TMR Data

In [ ]:
data_to_save = {
    'Voltage (V)': voltage_points,
    'TMR_Ratio': tmr_values,
    'TMR_Error': tmr_errors
}

df_results = pd.DataFrame(data_to_save)
output_path = PROJECT_ROOT / r"output/IV_H_scans/dataframes/b_scans"
filename = f"TMR_ratio_vs_V_{temperature}K.csv"
df_results.to_csv(output_path / filename, index=False)

print(f"Data successfully saved to: {filename}")
print("\nFirst 5 rows of the data saved:")
print(df_results.head())